### Tools
Models can request to call tools that perform task such as fetching data from a dsatabase searching the web or running code. Tools are paring of:
1. A schema including the name of the tool, a discription and/or argument defination(often a JSON shema)
2. a function or corouting to execute.

In [2]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [ ]:
from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"] =os.getenv("GROQ_API_KEY")


model=init_chat_model("groq:qwen/qwen3-32b")
response=model.invoke("why do parrots talk?")
response

AIMessage(content='<think>\nOkay, so the user is asking why parrots talk. Let me start by recalling what I know about parrots. Parrots are known for their ability to mimic human speech, but I need to make sure I cover the reasons behind this ability.\n\nFirst, I should mention their natural mimicry behavior. In the wild, parrots use mimicry to communicate with each other. Maybe they imitate other birds or sounds in their environment. But how does that translate to talking to humans?\n\nThen there\'s the aspect of social bonding. Parrots are highly social animals. In the wild, they form strong bonds with their flock. When they\'re with humans, they might mimic speech to bond with their human companions. It could be a way to get attention or to interact.\n\nI remember reading that parrots have a part of their brain called the "avian vocal learning system," which is similar to the areas in humans responsible for speech. This could explain their ability to learn and reproduce sounds. Maybe

In [9]:
from langchain.tools import tool

@tool
def get_weather(location:str)->str:
    """Get the weather at a location"""
    return F"It's sunny in {location}"

model_with_tool=model.bind_tools([get_weather])


In [10]:
response=model_with_tool.invoke("what is the weather of boston?")
print(response)

for tool_call in response.tool_calls:
    # view tools calls made by the llm
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

content='' additional_kwargs={'reasoning_content': "Okay, the user is asking for the weather in Boston. I need to use the get_weather function. Let me check the function's parameters. It requires a location, which is provided as Boston. So I should call get_weather with location set to Boston. Make sure the JSON is correctly formatted.\n", 'tool_calls': [{'id': '7cfswz609', 'function': {'arguments': '{"location":"boston"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 85, 'prompt_tokens': 154, 'total_tokens': 239, 'completion_time': 0.133262449, 'completion_tokens_details': {'reasoning_tokens': 60}, 'prompt_time': 0.013684167, 'prompt_tokens_details': None, 'queue_time': 0.050587473, 'total_time': 0.146946616}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_5cf921caa2', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019e7d5a-2584-7da3-b72c-25117fd2d59

### Tools Execution loop

In [11]:
# Step 1: Model genrates tool calls
messages=[{"role":"user","content":"what is the weather of bihar"}]
ai_msg= model_with_tool.invoke(messages)
messages.append(ai_msg)

# Step 2: Execute tools and collect result
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the genrated arguments
    tool_result=get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass results back to model for final response
final_response=model_with_tool.invoke(messages)
print(final_response)
# The current waether in bihar is 46C and sunny

content='The weather in Bihar is currently sunny.' additional_kwargs={'reasoning_content': "Okay, the user asked about the weather in Bihar. I called the get_weather function with location set to Bihar. The response came back saying it's sunny there. Now I need to relay that information clearly. Let me check if there's any additional details I should mention, but the function only provided the weather condition. So I'll just state that it's sunny in Bihar based on the response.\n"} response_metadata={'token_usage': {'completion_tokens': 93, 'prompt_tokens': 192, 'total_tokens': 285, 'completion_time': 0.166798008, 'completion_tokens_details': {'reasoning_tokens': 80}, 'prompt_time': 0.00779395, 'prompt_tokens_details': None, 'queue_time': 0.054801249, 'total_time': 0.174591958}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_5cf921caa2', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019e7d5a-34b6-75e0-8f50-262e7

In [12]:
messages

[{'role': 'user', 'content': 'what is the weather of bihar'},
 AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking about the weather in Bihar. Let me check the tools available. There\'s a function called get_weather that takes a location parameter. Since Bihar is a location, I should call that function with "Bihar" as the argument. I need to make sure the parameters are correctly formatted in JSON. Let me structure the tool call accordingly.\n', 'tool_calls': [{'id': '8811nsg8f', 'function': {'arguments': '{"location":"Bihar"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 98, 'prompt_tokens': 153, 'total_tokens': 251, 'completion_time': 0.139237915, 'completion_tokens_details': {'reasoning_tokens': 73}, 'prompt_time': 0.006952837, 'prompt_tokens_details': None, 'queue_time': 0.055454493, 'total_time': 0.146190752}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_5cf921caa2', 'servic